# KLTN — ABSA Token Merging · CHẠY FULL

Dữ liệu đầy đủ (2 448 / 304 / 312 mẫu), epoch mặc định, **21 biến thể × 3 seed ×
2 backbone**. Không có `--smoke` ở đâu trong notebook này.

> Chạy **`kaggle_smoke_test.ipynb`** trước ít nhất một lần. Nó mất 5–10 phút và
> bắt được lỗi môi trường trước khi bạn đốt hàng chục giờ GPU.

## Hai ràng buộc cứng của Kaggle

**1 · Thời gian — phiên tối đa 9–12 giờ.**
Lượt đầy đủ gồm 3 lượt train T5 ATE + 42 lượt train APC (stage `apc`) + 126 lượt
train APC (stage `multiseed`) = **171 lượt train**. Không nhét vừa một phiên.

**2 · Dung lượng — `/kaggle/working` chỉ 20 GB.**
Checkpoint APC (`best_model.pt`) nặng **0,43 GB**, checkpoint T5 nặng **0,83 GB**:

| Stage | Số checkpoint | Dung lượng |
|---|---:|---:|
| `multiseed` | 126 | 54,5 GB |
| `apc` | 42 | 18,2 GB |
| `ate` | 3 | 2,5 GB |
| `gas` | 1 | 0,8 GB |
| **Tổng** | **172** | **75,9 GB** |

→ **gấp gần 4 lần hạn mức.** Giữ hết checkpoint là chắc chắn hết đĩa giữa chừng.

## Chiến lược: cắt lát + dọn checkpoint + nối phiên

Điều làm cách này chạy được: `common/run_multiseed.py` **eval ngay sau mỗi lượt
train** rồi ghi vào `runs_multiseed/results_raw.csv`, và lần chạy sau nó bỏ qua
combo đã có trong CSV đó (`done_keys`, kiểm tra *trước* khi train). Nghĩa là
checkpoint của `multiseed` **xoá được ngay sau mỗi lát** — chỉ cần giữ
`results_raw.csv` là resume được.

Checkpoint của stage `apc` thì phải giữ tới khi `gold` + `triplet` + `results`
đọc xong, nên làm theo từng backbone rồi dọn.

| Phiên | Làm gì | Ước lượng |
|---|---|---|
| 1 | Giai đoạn 1 (ATE) + bắt đầu Giai đoạn 2 | ~9h |
| 2–3 | Giai đoạn 2 chạy tiếp | ~9h/phiên |
| 4 | Giai đoạn 3 (apc + gold + triplet) | ~9h |
| 5 | Giai đoạn 4 (gas + results + figures + report) | ~4h |

## Nối hai phiên trên Kaggle

`/kaggle/working` **không** tự sống qua phiên:

1. Cuối phiên: chạy cell cuối, rồi **Save Version → Save & Run All (Commit)**
2. Đợi commit xong → output thành một dataset
3. Phiên sau: **Add Input → Your Work → Notebook Output** (chọn version vừa commit)
4. Đặt `PREV_INPUT` ở cell 5 rồi chạy lại từ đầu

## Trước khi chạy

| Mục | Giá trị |
|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `GPU P100` |
| **Internet** | `On` |
| **Persistence** | `Files only` (nếu có) |


## 1 · Cấu hình

Bật sẵn trong panel bên phải: **Accelerator = GPU**, **Internet = On**.


In [ ]:
import os
from pathlib import Path

# ── Nguồn code ────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/hotuyen21pt/KLTN-Token-Merging.git"
BRANCH    = "tuyen"
REPO_NAME = "KLTN-Token-Merging"

# Repo private? Thêm Kaggle Secret tên GITHUB_TOKEN (Add-ons → Secrets).
USE_TOKEN = False

# ── Thư mục làm việc ──────────────────────────────────────────────────────
# /kaggle/working được commit lại sau phiên (~20 GB).
# /kaggle/temp bị xoá hết phiên → để cache model, khỏi tốn quota output.
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR  = WORK_ROOT / REPO_NAME
CACHE_DIR = Path("/kaggle/temp/hf") if Path("/kaggle").exists() else WORK_ROOT / ".hf"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["MPLBACKEND"] = "Agg"   # Kaggle headless

print(f"Repo dir : {REPO_DIR}")
print(f"HF cache : {CACHE_DIR}")
print(f"Branch   : {BRANCH}")


## 2 · Clone / pull code từ GitHub

Chạy lại được nhiều lần:

- **Lần đầu** → `git clone --branch <BRANCH> --single-branch`
- **Lần sau** → `remote set-url` → `fetch --all --prune --tags` →
  `checkout -B` → `reset --hard origin/<BRANCH>` → `clean -fd`

`reset --hard` đảm bảo code khớp *chính xác* GitHub. Kết quả thực nghiệm nằm
ngoài vùng git theo dõi (đã `.gitignore`) nên **không** bị xoá.


In [ ]:
import subprocess, sys, shutil

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh, in output theo thời gian thực (train chạy nhiều giờ)."""
    cmd = [str(c) for c in cmd]
    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, text=True, encoding="utf-8", errors="replace",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {' '.join(cmd)}")
    return proc.returncode


clone_url = REPO_URL
if USE_TOKEN:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    print("Dùng GITHUB_TOKEN từ Kaggle Secrets")

if (REPO_DIR / ".git").is_dir():
    print(f"Đã có repo tại {REPO_DIR} → cập nhật\n")
    sh(["git", "remote", "set-url", "origin", clone_url], cwd=REPO_DIR)
    sh(["git", "fetch", "--all", "--prune", "--tags"], cwd=REPO_DIR)
    sh(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
    sh(["git", "clean", "-fd"], cwd=REPO_DIR)
else:
    print(f"Chưa có repo → clone nhánh {BRANCH}\n")
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])

if USE_TOKEN:   # giấu token khỏi .git/config
    sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print()
sh(["git", "log", "--oneline", "-5"], cwd=REPO_DIR)
sh(["git", "status", "--short", "--branch"], cwd=REPO_DIR)
print(f"\nThư mục hiện tại: {os.getcwd()}")


## 3 · Cài thư viện

Image Kaggle đã có `torch`, `transformers`, `scikit-learn`, `matplotlib`, `pandas`.
Cell này bù những gói còn thiếu.

**`Levenshtein` là bắt buộc** — `src/normalization.py` import nó, thiếu là stage
`ate` / `ate_infer` / `gas` chết ngay với `ModuleNotFoundError`. Cell cài **từng
gói một lệnh pip riêng** (gộp chung thì một gói resolve hỏng làm pip bỏ cả lô)
và **dừng hẳn** nếu gói bắt buộc vẫn thiếu, thay vì chỉ cảnh báo rồi đi tiếp.

`pyabsa` là tuỳ chọn — không có thì các stage chính vẫn chạy.


In [ ]:
FULL_INSTALL = False   # True = cài đúng requirements.txt (lâu, dễ đụng torch của Kaggle)

# (tên pip, tên import, bắt buộc?)
PACKAGES = [
    ("python-Levenshtein>=0.25.0", "Levenshtein", True),
    ("seaborn",                    "seaborn",     True),
    ("tqdm",                       "tqdm",        True),
    ("pyabsa>=2.4.0,<3",           "pyabsa",      False),
]

if FULL_INSTALL:
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
else:
    for spec, _mod, required in PACKAGES:
        rc = sh([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
        if rc != 0:
            note = "  (BẮT BUỘC)" if required else "  (tuỳ chọn, bỏ qua được)"
            print(f"  [!] pip install {spec} → exit {rc}{note}")

print("\n" + "=" * 60)
missing = []
CHECK = PACKAGES + [
    ("torch", "torch", True), ("transformers", "transformers", True),
    ("scikit-learn", "sklearn", True), ("matplotlib", "matplotlib", True),
    ("pandas", "pandas", True),
]
for spec, mod, required in CHECK:
    try:
        m = __import__(mod)
        print(f"  {mod:<14}: {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        print(f"  {mod:<14}: {'THIẾU (BẮT BUỘC)' if required else 'thiếu (tuỳ chọn)'}"
              f" — {type(e).__name__}")
        if required:
            missing.append(spec)
print("=" * 60)

if missing:
    raise SystemExit(
        "DỪNG — thiếu gói bắt buộc.\nCài tay rồi RESTART kernel:\n"
        + "\n".join(f"  !pip install {s}" for s in missing)
    )
print("Đủ thư viện bắt buộc.")


## 4 · Kiểm tra môi trường


In [ ]:
sh(["nvidia-smi"], check=False)
print()
sh([sys.executable, "run_all.py", "--list"])


## 5 · Khôi phục trạng thái từ phiên trước

Phiên đầu tiên: để `PREV_INPUT = None` rồi chạy qua.


In [ ]:
# Trỏ vào output phiên trước đã attach làm input. Xem tên bằng: !ls /kaggle/input
PREV_INPUT = None      # ví dụ: "/kaggle/input/kltn-token-merging-full-v3"

# Đủ để resume — cố tình KHÔNG gồm best_model.pt của multiseed (54 GB)
RESTORE = [
    "runs_multiseed/results_raw.csv",        # ← state resume của multiseed
    "runs_multiseed/results_aggregated.csv",
    "runs_ate",                              # prediction ATE theo seed
    "checkpoints",                           # checkpoint T5 ATE
    "checkpoints_gas",
    "runs_joint", "runs_joint_t5",           # checkpoint APC (cho gold/triplet/results)
    "runs_bert_gold", "runs_gas", "reports",
]

if PREV_INPUT is None:
    print("PREV_INPUT = None → phiên đầu tiên, không khôi phục gì.")
    if Path("/kaggle/input").is_dir():
        found = sorted(p.name for p in Path("/kaggle/input").iterdir())
        print(f"Input đang attach: {found if found else '(trống)'}")
else:
    src_root = Path(PREV_INPUT)
    if not src_root.is_dir():
        raise SystemExit(f"Không thấy {src_root} — kiểm tra lại Add Input.")
    total = 0
    for rel in RESTORE:
        src = src_root / rel
        if not src.exists():
            continue
        dst = REPO_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            n = sum(f.stat().st_size for f in src.rglob("*") if f.is_file())
        else:
            shutil.copy2(src, dst)
            n = src.stat().st_size
        total += n
        print(f"  khôi phục {rel:<40} {n / 2**30:6.2f} GB")
    print(f"\nTổng: {total / 2**30:.2f} GB")

raw = REPO_DIR / "runs_multiseed" / "results_raw.csv"
if raw.is_file():
    import csv as _csv
    rows = list(_csv.DictReader(raw.open(encoding="utf-8")))
    keys = {(r["model_type"], r["config_id"], r["seed"]) for r in rows}
    print(f"\nTiến độ multiseed: {len(keys)}/126 combo đã xong "
          f"({len(keys) / 126 * 100:.0f}%) → sẽ được bỏ qua")


## 6 · Phạm vi + helper giới hạn thời gian

`run()` tự bỏ qua bước kế khi sắp hết giờ phiên, để notebook kịp chạy cell lưu
trạng thái ở cuối thay vì bị Kaggle giết ngang.

Muốn chạy gọn hơn thì thu hẹp `SEEDS` / `BACKBONES` / `GROUPS` ngay tại đây.


In [ ]:
import time

SESSION_HOURS = 8.5     # đặt thấp hơn hạn mức thật để còn thời gian commit

SEEDS     = ["42", "123", "456"]                  # gọn hơn: ["42"]
BACKBONES = ["bert", "t5"]                        # gọn hơn: ["bert"]
GROUPS    = ["resize", "compact", "pretome"]      # gọn hơn: ["resize"]

DEADLINE = time.time() + SESSION_HOURS * 3600


def time_left_min():
    return (DEADLINE - time.time()) / 60


def disk_free_gb(path="/kaggle/working"):
    target = path if Path(path).exists() else str(REPO_DIR)
    return shutil.disk_usage(target).free / 2**30


def run(*args, need_min=25):
    """Gọi run_all.py, tự bỏ qua nếu không còn đủ thời gian."""
    left = time_left_min()
    if left < need_min:
        print(f"[HẾT GIỜ] còn {left:.0f} phút < {need_min} → bỏ qua: {' '.join(args)}")
        return None
    print(f"\n{'#' * 78}")
    print(f"# còn {left:.0f} phút | đĩa trống {disk_free_gb():.1f} GB | {' '.join(args)}")
    print(f"{'#' * 78}")
    return sh([sys.executable, "run_all.py", *args, "--resume"], check=False)


def prune(pattern, why=""):
    """Xoá checkpoint đã dùng xong để không hết đĩa."""
    n = freed = 0
    for f in sorted(REPO_DIR.glob(pattern)):
        if f.is_file():
            freed += f.stat().st_size
            f.unlink()
            n += 1
    print(f"[dọn] xoá {n} file ({freed / 2**30:.1f} GB) — {why}")
    print(f"[dọn] đĩa trống còn {disk_free_gb():.1f} GB")


n_slices = len(SEEDS) * len(BACKBONES) * len(GROUPS)
n_runs = len(SEEDS) * len(BACKBONES) * (
    12 * ("resize" in GROUPS) + 6 * ("compact" in GROUPS) + 3 * ("pretome" in GROUPS))
print(f"Hạn phiên : {SESSION_HOURS} giờ (còn {time_left_min():.0f} phút)")
print(f"Đĩa trống : {disk_free_gb():.1f} GB")
print(f"Phạm vi   : {len(SEEDS)} seed × {len(BACKBONES)} backbone × {GROUPS}")
print(f"multiseed : {n_slices} lát, {n_runs} lượt train")


## 7 · Giai đoạn 1 — ATE (T5 GAS)

Train T5 ATE cho từng seed rồi sinh `test_ate_predictions.csv`. **Bắt buộc chạy
trước** mọi eval end-to-end. Repo không kèm checkpoint (đã `.gitignore`) nên
phiên đầu luôn phải train từ đầu.

Sinh ra `runs_ate/seed_<N>/test_predictions.csv` — nhờ đó stage `multiseed` ghép
cặp đúng ATE(seed N) ↔ APC(seed N) thay vì dùng chung một file cho mọi seed.


In [ ]:
run("--stages", "env", "--seeds", *SEEDS)
run("--stages", "ate", "--seeds", *SEEDS, need_min=90)
run("--stages", "ate_infer", "--seeds", *SEEDS, need_min=15)

print()
for s in SEEDS:
    f = REPO_DIR / "runs_ate" / f"seed_{s}" / "test_predictions.csv"
    print(f"  seed {s}: {'OK' if f.is_file() else 'THIẾU'}  {f}")
shared = REPO_DIR / "runs_ate" / "test_ate_predictions.csv"
print(f"  chung  : {'OK' if shared.is_file() else 'THIẾU'}  {shared}")


## 8 · Giai đoạn 2 — `multiseed` (126 lượt train, cắt lát)

Chạy từng lát `(seed × backbone × nhóm biến thể)` và **xoá checkpoint sau mỗi
lát**. Không xoá thì 54,5 GB checkpoint làm hết đĩa khoảng lát thứ 10.

Chạy lại nhiều phiên đều được — lát nào xong rồi sẽ tự `SKIP`.


In [ ]:
done, skipped = [], []

for seed in SEEDS:
    for bb in BACKBONES:
        for grp in GROUPS:
            tag = f"seed={seed} {bb} {grp}"
            rc = run("--stages", "multiseed",
                     "--seeds", seed, "--backbones", bb, "--variants", grp,
                     need_min=45)
            if rc is None:
                skipped.append(tag)
                continue
            done.append((tag, rc))
            # Checkpoint multiseed dùng một lần — eval đã chạy inline xong.
            prune("runs_multiseed/**/best_model.pt", f"đã eval xong {tag}")

print(f"\n{'=' * 78}")
print(f"Đã chạy {len(done)} lát, bỏ qua {len(skipped)} lát vì hết giờ")
for tag, rc in done:
    print(f"  {'OK ' if rc == 0 else 'rc=' + str(rc)}  {tag}")
if skipped:
    print("\nCòn lại cho phiên sau:")
    for tag in skipped:
        print(f"  - {tag}")


In [ ]:
# Tiến độ tổng
import csv as _csv
from collections import Counter

raw = REPO_DIR / "runs_multiseed" / "results_raw.csv"
if raw.is_file():
    rows = list(_csv.DictReader(raw.open(encoding="utf-8")))
    keys = {(r["model_type"], r["config_id"], r["seed"]) for r in rows}
    target = len(SEEDS) * len(BACKBONES) * 21
    print(f"{len(keys)}/{target} combo đã xong ({len(keys) / max(target, 1) * 100:.0f}%)")
    for bb, n in sorted(Counter(k[0] for k in keys).items()):
        print(f"  {bb:<6}: {n}")
    for sd, n in sorted(Counter(k[2] for k in keys).items()):
        print(f"  seed {sd:<4}: {n}")
else:
    print("Chưa có runs_multiseed/results_raw.csv")


## 9 · Giai đoạn 3 — `apc` + `gold` + `triplet` theo từng backbone

Stage `apc` train 21 biến thể vào `runs_joint*/` và **giữ** checkpoint, vì `gold`,
`triplet`, `results` cần đọc. 21 × 0,43 = 9,1 GB mỗi backbone — vừa đĩa nếu xong
backbone nào thì dọn backbone đó.

Chạy sau khi Giai đoạn 2 xong. Hết giờ thì để phiên sau.


In [ ]:
for bb in BACKBONES:
    rc = run("--stages", "apc", "--backbones", bb, "--seeds", SEEDS[0], need_min=120)
    if rc is None:
        print("Hết giờ — dừng Giai đoạn 3, phiên sau chạy tiếp.")
        break
    run("--stages", "gold", "--backbones", bb, need_min=20)
    run("--stages", "triplet", "--backbones", bb, need_min=20)

    # Bỏ comment nếu sắp hết đĩa. LƯU Ý: `results` ở Giai đoạn 4 cũng cần
    # checkpoint này, nên chỉ dọn khi đã chạy `results` hoặc chấp nhận bỏ nó.
    # prune(f"{'runs_joint' if bb == 'bert' else 'runs_joint_t5'}/*/best_model.pt",
    #       f"xong gold+triplet cho {bb}")

print(f"\nĐĩa trống: {disk_free_gb():.1f} GB")


## 10 · Giai đoạn 4 — GAS, results, hình, báo cáo

- `gas` — train GAS một bước (sinh thẳng bộ ba) làm baseline đối chứng. Độc lập
  với ATE→APC, bỏ qua được.
- `results` — eval trên `results.csv`; **cần** checkpoint APC trong `runs_joint*/`
  nên phải chạy trước khi dọn ở Giai đoạn 3.
- `multiseed --no-train` — **bắt buộc**: gộp lại toàn bộ 8 bảng luận văn từ
  `results_raw.csv` đã tích luỹ, vì mỗi lát ở Giai đoạn 2 chỉ sinh bảng cho nhóm
  biến thể của riêng nó.


In [ ]:
run("--stages", "gas", "--seeds", SEEDS[0], need_min=60)
run("--stages", "results", need_min=30)
run("--stages", "figures", need_min=10)

# Gộp lại toàn bộ bảng luận văn từ mọi lát đã chạy
run("--stages", "multiseed", "--no-train",
    "--seeds", *SEEDS, "--backbones", *BACKBONES, need_min=5)

run("--stages", "report", need_min=5)


## 11 · Xem kết quả


In [ ]:
from IPython.display import Markdown, display

report = REPO_DIR / "reports" / "REPORT.md"
if report.is_file():
    txt = report.read_text(encoding="utf-8")
    if "Lượt CHẠY THỬ" in txt:
        print("⚠️  Đây là báo cáo của lượt SMOKE, không phải lượt full.")
    display(Markdown(txt))
else:
    print(f"Chưa có {report} — chạy stage `report`.")


In [ ]:
tables = REPO_DIR / "runs_multiseed" / "thesis_tables.txt"
if tables.is_file():
    print(tables.read_text(encoding="utf-8"))
else:
    print(f"Chưa có {tables}")


## 12 · Lưu trạng thái cho phiên sau

Chạy cell này **trước khi** bấm Save Version. Nó liệt kê dung lượng những thứ cần
giữ và cảnh báo nếu vượt hạn mức output, đồng thời tạo một bản zip nhẹ chỉ gồm
số liệu (luôn tải về được kể cả khi checkpoint bị dọn).

Sau đó: **Save Version → Save & Run All (Commit)** → phiên mới
**Add Input → Your Work → Notebook Output** → đặt `PREV_INPUT` ở cell 5.


In [ ]:
KEEP = [
    "runs_multiseed/results_raw.csv",
    "runs_multiseed/results_aggregated.csv",
    "runs_multiseed/thesis_tables.txt",
    "runs_multiseed/results_summary.txt",
    "runs_ate", "runs_bert_gold", "runs_gas", "reports",
    "checkpoints",        # T5 ATE — train lại rất tốn, nên giữ
    "checkpoints_gas",
    "runs_joint", "runs_joint_t5",
]

print(f"{'đường dẫn':<42} {'GB':>7}")
print("-" * 60)
total = 0
for rel in KEEP:
    p_ = REPO_DIR / rel
    if not p_.exists():
        print(f"{rel:<42} {'—':>7}  (chưa có)")
        continue
    size = (sum(f.stat().st_size for f in p_.rglob("*") if f.is_file())
            if p_.is_dir() else p_.stat().st_size)
    total += size
    print(f"{rel:<42} {size / 2**30:7.2f}")
print("-" * 60)
print(f"{'TỔNG':<42} {total / 2**30:7.2f} GB")

if total / 2**30 > 19:
    print("\n⚠️  Vượt ~20 GB hạn mức output Kaggle. Dọn bớt rồi chạy lại cell này:")
    print("   prune('runs_joint/*/best_model.pt',    'giai phong cho commit')")
    print("   prune('runs_joint_t5/*/best_model.pt', 'giai phong cho commit')")
else:
    print("\nVừa hạn mức. Bấm Save Version → Save & Run All (Commit).")

# Zip nhẹ chỉ gồm số liệu
import zipfile
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base = Path("/kaggle/working") if Path("/kaggle/working").exists() else REPO_DIR
out_zip = base / f"kltn_results_{stamp}.zip"
n = 0
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for pat in ["runs_multiseed/*.csv", "runs_multiseed/*.txt",
                "runs_ate/**/*.csv", "runs_ate/*.txt",
                "runs_joint*/experiment_results_joint.*",
                "runs_joint*/*/meta.json",
                "runs_bert_gold/**/*.csv", "runs_gas/*",
                "reports/**/*", "thesis/figures/*"]:
        for f in sorted(REPO_DIR.glob(pat)):
            if f.is_file() and f.suffix not in {".pt", ".bin", ".safetensors"}:
                zf.write(f, f.relative_to(REPO_DIR))
                n += 1
print(f"\nZip số liệu: {n} file → {out_zip} ({out_zip.stat().st_size / 2**20:.1f} MB)")
